# Wiegelmann & Sakurai (2012) — Solar Force-Free Magnetic Fields / 태양 무력장

**Paper.** Wiegelmann, T. & Sakurai, T. (2012). *Solar Force-Free Magnetic Fields*. Living Reviews in Solar Physics, 9, 5. DOI: [10.12942/lrsp-2012-5](https://doi.org/10.12942/lrsp-2012-5)

## Goal / 목표

**EN.** Demonstrate the four central concepts of the review through small, runnable numerical experiments:
1. Force-free condition $\mathbf{j}\times\mathbf{B}=0\Rightarrow\nabla\times\mathbf{B}=\alpha\mathbf{B}$ — verify on a known toy field.
2. Linear (constant-$\alpha$) vs. nonlinear ($\alpha$ varying along field line) force-free fields.
3. Potential field above a magnetogram via the Green-function (Schmidt 1964) integral.
4. Wiegelmann's optimization toy — minimize $L=\int(B^{-2}|(\nabla\times B)\times B|^2+|\nabla\cdot B|^2)dV$ in 1D.

**KR.** 본 리뷰의 핵심 개념 네 가지를 작은 수치 실험으로 시연합니다:
1. 무력장 조건 $\mathbf{j}\times\mathbf{B}=0\Rightarrow\nabla\times\mathbf{B}=\alpha\mathbf{B}$를 알려진 toy 자기장에서 검증.
2. 선형(상수-$\alpha$) 대 비선형($\alpha$가 자기력선을 따라 변하는) 무력장 비교.
3. Green 함수 적분(Schmidt 1964)을 통한 자기도 위 퍼텐셜장 계산.
4. Wiegelmann 최적화 toy — 1D에서 $L=\int(B^{-2}|(\nabla\times B)\times B|^2+|\nabla\cdot B|^2)dV$ 최소화.

Kernel: `python3` (conda env `study-with-ai`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

print('NumPy:', np.__version__)

## Part 1 — Force-free condition on the Lundquist (1950) flux rope / Lundquist 자속관에서 무력장 조건 검증

**EN.** Lundquist's solution is constant-$\alpha$ in cylindrical coordinates: $B_z=B_0 J_0(\alpha r)$, $B_\varphi=B_0 J_1(\alpha r)$, $B_r=0$. We verify (i) $\nabla\cdot\mathbf{B}=0$, (ii) $\nabla\times\mathbf{B}=\alpha\mathbf{B}$, (iii) $\mathbf{j}\times\mathbf{B}=0$.

**KR.** Lundquist 해는 원통좌표에서 상수-$\alpha$: $B_z=B_0 J_0(\alpha r)$, $B_\varphi=B_0 J_1(\alpha r)$, $B_r=0$. (i) $\nabla\cdot\mathbf{B}=0$, (ii) $\nabla\times\mathbf{B}=\alpha\mathbf{B}$, (iii) $\mathbf{j}\times\mathbf{B}=0$를 검증합니다.

In [ ]:
from scipy.special import j0, j1

def lundquist_field(r, phi, z, alpha=1.0, B0=1.0):
    """Compute Cartesian components of Lundquist's constant-alpha flux rope.

    Args:
        r: Radial coordinate array.
        phi: Azimuthal coordinate array.
        z: Vertical coordinate array (unused; field is z-invariant).
        alpha: Force-free parameter.
        B0: Field strength scale.

    Returns:
        Tuple (Bx, By, Bz) of Cartesian field components.
    """
    Bz = B0 * j0(alpha * r)
    Bphi = B0 * j1(alpha * r)
    Bx = -Bphi * np.sin(phi)
    By = Bphi * np.cos(phi)
    return Bx, By, Bz

# Sample on a 3D Cartesian grid.
n = 64
L = 4.0
x = np.linspace(-L, L, n)
y = np.linspace(-L, L, n)
z = np.linspace(0, L, n // 2)
dx = x[1] - x[0]
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
R = np.sqrt(X**2 + Y**2)
PHI = np.arctan2(Y, X)

alpha_true = 0.5
Bx, By, Bz = lundquist_field(R, PHI, Z, alpha=alpha_true, B0=1.0)
print('Field shape:', Bx.shape)

In [ ]:
def curl(Bx, By, Bz, dx):
    """Second-order central-difference curl on a uniform Cartesian grid.

    Args:
        Bx, By, Bz: Field components on a regular grid.
        dx: Grid spacing (assumed uniform isotropic).

    Returns:
        Tuple (Jx, Jy, Jz) representing curl B.
    """
    dBz_dy, dBz_dx = np.gradient(Bz, dx, axis=(1, 0))
    dBy_dz = np.gradient(By, dx, axis=2)
    dBx_dz = np.gradient(Bx, dx, axis=2)
    dBy_dx = np.gradient(By, dx, axis=0)
    dBx_dy = np.gradient(Bx, dx, axis=1)
    Jx = dBz_dy - dBy_dz
    Jy = dBx_dz - dBz_dx
    Jz = dBy_dx - dBx_dy
    return Jx, Jy, Jz

def divergence(Bx, By, Bz, dx):
    """Second-order central-difference divergence."""
    return (np.gradient(Bx, dx, axis=0)
            + np.gradient(By, dx, axis=1)
            + np.gradient(Bz, dx, axis=2))

Jx, Jy, Jz = curl(Bx, By, Bz, dx)
divB = divergence(Bx, By, Bz, dx)

B2 = Bx**2 + By**2 + Bz**2 + 1e-20
# alpha estimated from curl B = alpha B (use B dot curl B / B^2)
alpha_est = (Bx * Jx + By * Jy + Bz * Jz) / B2

# Lorentz force per unit B^2.
Lx = Jy * Bz - Jz * By
Ly = Jz * Bx - Jx * Bz
Lz = Jx * By - Jy * Bx
Lorentz = np.sqrt(Lx**2 + Ly**2 + Lz**2)

# Mask near rope axis where derivatives are dominated by finite-difference noise.
mask = (R > 0.5) & (R < 3.0) & (Z > 0.2) & (Z < L - 0.2)

print(f'true alpha            = {alpha_true:.4f}')
print(f'mean estimated alpha  = {np.mean(alpha_est[mask]):.4f}')
print(f'max |div B|/<|B|>     = {np.max(np.abs(divB[mask]))/np.mean(np.sqrt(B2[mask])):.2e}')
print(f'mean |J x B|/<B^2>    = {np.mean(Lorentz[mask])/np.mean(B2[mask]):.2e}')

**EN.** The estimated $\alpha$ matches the true value (small finite-difference error away from the axis), $\nabla\cdot\mathbf{B}\approx 0$, and the Lorentz force is small — confirming Lundquist is a force-free field with the prescribed $\alpha$.

**KR.** 추정된 $\alpha$가 참값과 일치(축에서 떨어진 영역에서 유한차분 오차 작음), $\nabla\cdot\mathbf{B}\approx 0$, Lorentz 힘이 매우 작음 — Lundquist 해가 지정된 $\alpha$로 무력장임을 확인.

## Part 2 — Linear vs. Nonlinear force-free: $\alpha$ along field lines / 선형 vs 비선형

**EN.** A *linear* force-free field has $\alpha=$ constant. A *nonlinear* force-free field has $\alpha$ varying between field lines but constant along each (because $\mathbf{B}\cdot\nabla\alpha=0$). We construct a quasi-NLFFF (Low-Lou-flavored) toy: one Seehafer mode with spatially varying $\alpha(z)$ — strictly speaking *not* force-free, but illustrative of how $\alpha$ should and should not vary.

**KR.** *선형* 무력장은 $\alpha$가 공간 상수. *비선형* 무력장은 자기력선 사이에서는 $\alpha$가 다르나 한 자기력선 위에서는 일정($\mathbf{B}\cdot\nabla\alpha=0$). Seehafer 단일 모드에 $\alpha(z)$가 변하는 toy 자기장을 비교 — 엄밀히는 force-free가 아니나, $\alpha$가 어떻게 변해야/변해서는 안 되는지 시연용.

In [ ]:
def seehafer_lfff(x, y, z, alpha=0.0, kx=np.pi, ky=np.pi, B0=1.0):
    """Single-mode Seehafer linear force-free field.

    For Bz(x,y,0) = B0 sin(kx x) sin(ky y), the solution is
        Bz = B0 exp(-r z) sin(kx x) sin(ky y)
        Bx = (B0 / lam) * exp(-r z) * (alpha*ky*sin*cos - r*kx*cos*sin)
        By = -(B0 / lam) * exp(-r z) * (alpha*kx*cos*sin + r*ky*sin*cos)
    with lam = kx^2+ky^2 and r = sqrt(lam - alpha^2). Requires alpha^2 < lam.
    """
    lam = kx**2 + ky**2
    if alpha**2 >= lam:
        raise ValueError('alpha^2 must be < kx^2+ky^2 for decaying solution.')
    r = np.sqrt(lam - alpha**2)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    sx, cx = np.sin(kx * X), np.cos(kx * X)
    sy, cy = np.sin(ky * Y), np.cos(ky * Y)
    decay = np.exp(-r * Z)
    Bz = B0 * decay * sx * sy
    Bx = (B0 / lam) * decay * (alpha * ky * sx * cy - r * kx * cx * sy)
    By = -(B0 / lam) * decay * (alpha * kx * cx * sy + r * ky * sx * cy)
    return Bx, By, Bz, r

# Domain.
n2 = 48
x2 = np.linspace(0, 1, n2)
y2 = np.linspace(0, 1, n2)
z2 = np.linspace(0, 1, n2)
dx2 = x2[1] - x2[0]
kx = ky = np.pi
lam = kx**2 + ky**2

alphas = [0.0, 1.5, 3.0, 4.0]
decay_rates = []
for a in alphas:
    _, _, _, r = seehafer_lfff(x2, y2, z2, alpha=a, kx=kx, ky=ky)
    decay_rates.append(r)
    print(f'alpha = {a:.2f} -> r = sqrt(lam - alpha^2) = {r:.4f}')
print(f'lam_max = pi^2 + pi^2 = {lam:.4f},  alpha_max = {np.sqrt(lam):.4f}')

In [ ]:
# Plot Bz at z=0.3 for different alpha to show how field lines twist.
fig, axes = plt.subplots(1, len(alphas), figsize=(14, 3.5), sharex=True, sharey=True)
for ax, a in zip(axes, alphas):
    Bx_, By_, Bz_, _ = seehafer_lfff(x2, y2, np.array([0.3]), alpha=a, kx=kx, ky=ky)
    Bzslice = Bz_[:, :, 0]
    Bxslice = Bx_[:, :, 0]
    Byslice = By_[:, :, 0]
    pc = ax.imshow(Bzslice.T, extent=[0, 1, 0, 1], origin='lower', cmap='RdBu_r', vmin=-1, vmax=1)
    step = max(1, n2 // 12)
    ax.quiver(x2[::step], y2[::step], Bxslice[::step, ::step].T, Byslice[::step, ::step].T,
              color='k', alpha=0.7)
    ax.set_title(f'alpha = {a:.1f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
fig.suptitle('Seehafer LFFF: Bz at z=0.3 with horizontal-field arrows / 선형 무력장 슬라이스')
plt.tight_layout(); plt.show()

In [ ]:
# Verify: for LFFF, alpha estimated from B . curl B / B^2 should equal the prescribed alpha at every point.
Bx_, By_, Bz_, _ = seehafer_lfff(x2, y2, z2, alpha=2.0, kx=kx, ky=ky)
Jx_, Jy_, Jz_ = curl(Bx_, By_, Bz_, dx2)
B2_ = Bx_**2 + By_**2 + Bz_**2 + 1e-20
alpha_est_lfff = (Bx_ * Jx_ + By_ * Jy_ + Bz_ * Jz_) / B2_

interior = (slice(2, -2), slice(2, -2), slice(2, -2))
print(f'Prescribed alpha     = 2.0')
print(f'Estimated alpha mean = {alpha_est_lfff[interior].mean():.4f}')
print(f'Estimated alpha std  = {alpha_est_lfff[interior].std():.4f}')
print('=> small std confirms a single global alpha (LFFF) / 단일 alpha 확인.')

In [ ]:
# Now an artificial NLFFF-like superposition: two Seehafer modes with different alpha.
# This is NOT force-free in general, but illustrates spatially varying alpha estimate.
Bx1, By1, Bz1, _ = seehafer_lfff(x2, y2, z2, alpha=1.5, kx=kx, ky=ky, B0=1.0)
Bx2, By2, Bz2, _ = seehafer_lfff(x2, y2, z2, alpha=3.5, kx=2*kx, ky=2*ky, B0=0.4)
Bxs = Bx1 + Bx2
Bys = By1 + By2
Bzs = Bz1 + Bz2
Jxs, Jys, Jzs = curl(Bxs, Bys, Bzs, dx2)
B2s = Bxs**2 + Bys**2 + Bzs**2 + 1e-20
alpha_est_nlfff = (Bxs * Jxs + Bys * Jys + Bzs * Jzs) / B2s

fig, ax = plt.subplots()
ax.hist(alpha_est_lfff[interior].ravel(), bins=60, alpha=0.6, label='LFFF: single $\\alpha$')
ax.hist(alpha_est_nlfff[interior].ravel(), bins=60, alpha=0.6, label='NLFFF-like: $\\alpha$ varies')
ax.set_xlabel(r'$\alpha = (B\cdot\nabla\times B)/B^2$'); ax.set_ylabel('voxel count')
ax.set_title('Distribution of estimated alpha / 추정 $\\alpha$ 분포')
ax.legend(); plt.tight_layout(); plt.show()

**EN.** LFFF gives a delta-function $\alpha$ histogram (single value). The NLFFF-like field shows a broad distribution — exactly the empirical situation that motivates NLFFF reconstruction (Pevtsov, Wheatland).

**KR.** LFFF는 단일 $\alpha$ 값(델타). NLFFF-스타일 자기장은 분포가 넓음 — 이것이 NLFFF 재구성을 정당화하는 실험적 상황(Pevtsov, Wheatland).

## Part 3 — Potential field above a magnetogram (Green function) / 자기도 위 퍼텐셜장

**EN.** Schmidt's (1964) Green function gives the upper-half-space potential field for prescribed $B_z(x,y,0)$:
$$
B_z(x,y,z) = \frac{z}{2\pi}\iint \frac{B_z(x',y',0)}{[(x-x')^2+(y-y')^2+z^2]^{3/2}}\,dx'\,dy'.
$$
We implement this with `np.trapezoid` for a synthetic dipole-pair magnetogram.

**KR.** Schmidt(1964) Green 함수는 주어진 광구 $B_z(x,y,0)$로부터 상반공간 퍼텐셜장 $B_z(x,y,z)$를 줍니다. 인공 쌍극자 자기도에 대해 `np.trapezoid`로 구현.

In [ ]:
def synthetic_bipole(nx=64, ny=64, L=10.0, sigma=1.0, sep=3.0, B0=100.0):
    """Construct a Gaussian bipolar magnetogram on a square grid.

    Args:
        nx, ny: Number of grid points along x and y.
        L: Half-width of the domain.
        sigma: Gaussian spot width (Mm).
        sep: Separation between positive and negative spots (Mm).
        B0: Peak field strength (G).

    Returns:
        Tuple (x, y, Bz) with axes and 2D Bz array.
    """
    x = np.linspace(-L, L, nx)
    y = np.linspace(-L, L, ny)
    X, Y = np.meshgrid(x, y, indexing='ij')
    pos = B0 * np.exp(-((X - sep / 2) ** 2 + Y ** 2) / (2 * sigma ** 2))
    neg = -B0 * np.exp(-((X + sep / 2) ** 2 + Y ** 2) / (2 * sigma ** 2))
    return x, y, pos + neg

x_mg, y_mg, Bz0 = synthetic_bipole(nx=80, ny=80, L=10.0, sigma=1.2, sep=4.0, B0=200.0)

fig, ax = plt.subplots()
im = ax.imshow(Bz0.T, extent=[x_mg[0], x_mg[-1], y_mg[0], y_mg[-1]],
               origin='lower', cmap='RdBu_r', vmin=-200, vmax=200)
ax.set_title('Synthetic photospheric Bz (G) / 인공 광구 Bz')
ax.set_xlabel('x [Mm]'); ax.set_ylabel('y [Mm]')
plt.colorbar(im, ax=ax, label='Bz [G]'); plt.tight_layout(); plt.show()

In [ ]:
def potential_bz_from_magnetogram(x, y, Bz0, z_levels):
    """Compute Bz at heights z_levels from photospheric Bz via Schmidt's Green function.

    Uses np.trapezoid for the 2D quadrature.

    Args:
        x, y: 1D coordinate axes of the magnetogram.
        Bz0: 2D array with Bz(x_i, y_j, 0).
        z_levels: 1D array of heights to evaluate.

    Returns:
        3D array with Bz at (x, y, z_levels).
    """
    Xp, Yp = np.meshgrid(x, y, indexing='ij')
    nx, ny = Bz0.shape
    nz = len(z_levels)
    Bz = np.zeros((nx, ny, nz))
    for k, z in enumerate(z_levels):
        for i in range(nx):
            for j in range(ny):
                R3 = ((x[i] - Xp) ** 2 + (y[j] - Yp) ** 2 + z ** 2) ** 1.5
                integrand = Bz0 / R3
                # 2D trapezoidal integration via np.trapezoid.
                inner = np.trapezoid(integrand, x=y, axis=1)
                outer = np.trapezoid(inner, x=x)
                Bz[i, j, k] = (z / (2 * np.pi)) * outer
    return Bz

# Use a small set of heights so this runs fast on an 80x80 magnetogram.
z_levels = np.array([0.5, 1.5, 3.0, 6.0])
Bz_pot = potential_bz_from_magnetogram(x_mg, y_mg, Bz0, z_levels)
print('Bz(z) shape:', Bz_pot.shape)

In [ ]:
fig, axes = plt.subplots(1, len(z_levels), figsize=(14, 3.5), sharex=True, sharey=True)
for ax, k, z in zip(axes, range(len(z_levels)), z_levels):
    im = ax.imshow(Bz_pot[:, :, k].T, extent=[x_mg[0], x_mg[-1], y_mg[0], y_mg[-1]],
                   origin='lower', cmap='RdBu_r', vmin=-100, vmax=100)
    ax.set_title(f'z = {z:.1f} Mm')
    ax.set_xlabel('x [Mm]')
axes[0].set_ylabel('y [Mm]')
fig.suptitle('Potential Bz at increasing heights (decay & broadening) / 높이별 퍼텐셜 Bz')
plt.tight_layout(); plt.show()

# Confirm decay: peak Bz vs z.
peaks = np.max(np.abs(Bz_pot), axis=(0, 1))
for z, p in zip(z_levels, peaks):
    print(f'z = {z:.1f} Mm: peak |Bz| = {p:.2f} G')

**EN.** As height increases, $|B_z|$ peak decays and the structure broadens — the classic signature of a potential field above an isolated bipole.

**KR.** 높이가 증가할수록 $|B_z|$ 최대값이 감쇠하고 구조가 넓어집니다 — 고립 쌍극자 위 퍼텐셜장의 전형적 특성.

## Part 4 — Wiegelmann optimization toy in 1D / 1D Wiegelmann 최적화 toy

**EN.** Wiegelmann's NLFFF method minimizes
$$L = \int_V \left[\,\frac{|(\nabla\times\mathbf{B})\times\mathbf{B}|^2}{B^2} + |\nabla\cdot\mathbf{B}|^2\right]\,dV.$$
We illustrate gradient descent in a tiny 1D analogue: minimize $L(\alpha) = \int_0^L (B_z'(z)+\alpha B_y(z))^2 + (B_y'(z)-\alpha B_z(z))^2 \,dz$ for a constant-$\alpha$ trial field, fitting $\alpha$ from a measured $\nabla\times\mathbf{B}$, $\mathbf{B}$ pair.

**KR.** Wiegelmann NLFFF는 $L=\int_V[B^{-2}|(\nabla\times\mathbf{B})\times\mathbf{B}|^2+|\nabla\cdot\mathbf{B}|^2]dV$를 최소화합니다. 1D 축약: 상수 $\alpha$ trial 자기장에 대해 $L(\alpha)=\int_0^L (B_z'+\alpha B_y)^2+(B_y'-\alpha B_z)^2 dz$ 최소화로 $\alpha$ 적합.

In [ ]:
def make_1d_lfff_field(z, alpha=1.0, B0=1.0):
    """Construct a 1D linear force-free field with prescribed alpha.

    The simplest non-trivial 1D LFFF that satisfies curl B = alpha B has
        Bx(z) = 0
        By(z) = B0 * sin(alpha * z)
        Bz(z) = B0 * cos(alpha * z)
    so that By' = alpha*Bz and -Bz' = alpha*By.
    """
    return B0 * np.sin(alpha * z), B0 * np.cos(alpha * z)

def loss_alpha(alpha, z, By, Bz):
    """Compute the 1D Wiegelmann-style residual L(alpha) by trapezoidal integration."""
    dBy_dz = np.gradient(By, z)
    dBz_dz = np.gradient(Bz, z)
    res1 = dBz_dz + alpha * By  # should be 0
    res2 = dBy_dz - alpha * Bz  # should be 0
    return np.trapezoid(res1 ** 2 + res2 ** 2, z)

z = np.linspace(0, 1, 200)
alpha_true = 2.5
By_obs, Bz_obs = make_1d_lfff_field(z, alpha=alpha_true)
# Add a little noise to mimic observation error.
By_noisy = By_obs + 0.02 * rng.normal(size=z.size)
Bz_noisy = Bz_obs + 0.02 * rng.normal(size=z.size)

alphas_scan = np.linspace(-5, 5, 401)
Lvals = np.array([loss_alpha(a, z, By_noisy, Bz_noisy) for a in alphas_scan])
alpha_min = alphas_scan[np.argmin(Lvals)]
print(f'true alpha     = {alpha_true:.4f}')
print(f'recovered min  = {alpha_min:.4f} (grid resolution {alphas_scan[1]-alphas_scan[0]:.3f})')

fig, ax = plt.subplots()
ax.plot(alphas_scan, Lvals)
ax.axvline(alpha_true, color='red', ls='--', label=f'true $\\alpha={alpha_true}$')
ax.axvline(alpha_min, color='green', ls=':', label=f'min $L$ at $\\alpha={alpha_min:.2f}$')
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$L(\alpha)$')
ax.set_title('Wiegelmann-style optimization landscape (1D toy) / 1D 최적화 풍경')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Gradient-descent recovery of alpha (more interesting in 3D, but the principle is identical).
def grad_descent_alpha(z, By, Bz, alpha0=0.0, lr=0.05, n_steps=200):
    """Steepest-descent loop minimizing L(alpha) with finite-difference gradient."""
    a = alpha0
    history = [a]
    losses = [loss_alpha(a, z, By, Bz)]
    eps = 1e-3
    for _ in range(n_steps):
        gL = (loss_alpha(a + eps, z, By, Bz) - loss_alpha(a - eps, z, By, Bz)) / (2 * eps)
        a -= lr * gL
        history.append(a)
        losses.append(loss_alpha(a, z, By, Bz))
    return np.array(history), np.array(losses)

hist, losses = grad_descent_alpha(z, By_noisy, Bz_noisy, alpha0=0.5, lr=0.05, n_steps=80)
print(f'final alpha (descent) = {hist[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist); axes[0].axhline(alpha_true, color='red', ls='--', label='true')
axes[0].set_xlabel('step'); axes[0].set_ylabel(r'$\alpha$')
axes[0].set_title('Convergence of $\\alpha$ / $\\alpha$ 수렴'); axes[0].legend()
axes[1].plot(losses); axes[1].set_xlabel('step'); axes[1].set_ylabel(r'$L$')
axes[1].set_title('Loss vs step / 손실 변화'); axes[1].set_yscale('log')
plt.tight_layout(); plt.show()

**EN.** Gradient descent on the Wiegelmann-style functional recovers the true $\alpha$ within a fraction of a percent, despite 2% additive noise on the observed field components. In real 3D NLFFF the unknown is the entire vector field $\mathbf{B}(\mathbf{r})$, the loss includes a divergence term, and the boundary is held fixed at the preprocessed magnetogram — but the underlying optimization principle is exactly this.

**KR.** 2% 잡음에도 불구하고 Wiegelmann-스타일 functional의 경사하강은 참 $\alpha$를 백분의 일 수준 정확도로 복원합니다. 실제 3D NLFFF에서는 미지수가 전체 벡터장 $\mathbf{B}(\mathbf{r})$이고, 손실에 발산 항이 포함되며, 경계가 preprocessing된 자기도로 고정되지만, 근본 원리는 이와 동일합니다.

## Summary / 요약

**EN.** We have:
1. Verified the force-free condition on Lundquist's analytic flux rope (constant-$\alpha$, $\mathbf{j}\times\mathbf{B}\approx 0$).
2. Compared LFFF (single $\alpha$) to a synthetic NLFFF-like superposition (broad $\alpha$ distribution) — the empirical motivation for NLFFF.
3. Computed the potential field above a synthetic bipolar magnetogram with Schmidt's Green-function integral and confirmed height decay.
4. Demonstrated the Wiegelmann optimization principle in 1D: minimizing the force-free residual recovers the true $\alpha$ from noisy data.

**KR.** 본 노트북에서 우리는:
1. Lundquist 자속관에서 무력장 조건 검증(상수-$\alpha$, $\mathbf{j}\times\mathbf{B}\approx 0$).
2. LFFF(단일 $\alpha$)와 NLFFF-스타일 중첩($\alpha$ 분포 넓음) 비교 — NLFFF의 실험적 동기.
3. Schmidt Green 함수 적분으로 인공 쌍극자 자기도 위 퍼텐셜장 계산, 높이 감쇠 확인.
4. 1D에서 Wiegelmann 최적화 원리 시연: 무력장 잔차 최소화로 잡음 데이터에서 참 $\alpha$ 복원.